In [58]:
import numpy as np

In [59]:
sclf_list = ["xdawn","lr","eegnet","atteca","eegconf","eegnex"]
param_list = ["ori", "tt", "csub", "in", "cin", "ll"]
str_dict = {"xdawn": "xDAWN+LR", "lr": "LR", "eegnet": "EEGNet", "atteca": "AttBaseNet", "eegconf": "EEGConf", "eegnex": "EEGNeX"}

In [60]:
def trans_arr(param, out_list):
    arr = np.array(out_list)
    if param in ("cin", "csub"):
        return arr
    elif param == "ll":
        return arr.mean(axis=(2, 4))
    return arr.mean(axis=2)

In [61]:
for param in param_list:
    is_split = param in ("in", "cin")
    suffixes = ("p", "i") if is_split else (None,)
    out = {s: {} for s in suffixes} if is_split else {}
    for sclf in sclf_list:
        key = str_dict[sclf]
        for suffix in suffixes:
            files = [
                f"multi/acc/sub/{sclf}_{param}_{sub}"
                f"{'_' + suffix if suffix else ''}.npy"
                for sub in range(20)
            ]
            ddlist = [np.load(f) for f in files]
            arr = trans_arr(param, ddlist)
            if is_split:
                out[suffix][key] = arr
            else:
                out[key] = arr
    np.save(f"multi/acc/comb/acc_{param}.npy", out)

In [62]:
for param in ["ori", "tt", "csub", "ll"]:
    acc = np.load(f"multi/acc/comb/acc_{param}.npy", allow_pickle=True).item()
    print(f"Shape of {param}: {acc['LR'].shape}")

Shape of ori: (20, 2, 2)
Shape of tt: (20, 2, 35)
Shape of csub: (20, 2, 2)
Shape of ll: (20, 2, 8, 2)
